In [141]:
import polars as pl
from pathlib import Path

itu_path = Path("data_itu.csv")
cepal_path = Path("data_cepal.csv")

df_itu = pl.read_csv(itu_path)
df_cepal = pl.read_csv(cepal_path)

print("CEPAL shape:", df_cepal.shape)
print("ITU shape:", df_itu.shape)

CEPAL shape: (870, 8)
ITU shape: (4110, 13)


## Paso 1: Limpieza de Columnas y Reducción de Dimensionalidad

Se aplica una **reducción de dimensionalidad** eliminando columnas administrativas, identificadores internos y metadatos que no aportan valor predictivo al análisis. En esta fase, se filtra los datos de la CEPAL para conservar únicamente el segmento "Total". Esta decisión es crítica para homogeneizar la unidad de análisis entre ambos conjuntos de datos, lo que permite capturar el comportamiento macroeconómico del uso de internet sin el "ruido" que introduciría la desagregación por rangos de edad.

In [142]:
cols_to_remove_cepal = [
    "indicator",
    "unit",
    "notes_ids",
    "source_id",
    "Grupos etarios Uso Internet"
]

df_cepal = df_cepal.filter(pl.col("Grupos etarios Uso Internet") == "Total")

df_cepal = df_cepal.drop(cols_to_remove_cepal)

cols_to_remove_itu = [
    "seriesID",
    "seriesCode",
    "seriesName",
    "seriesParent",
    "seriesUnits",
    "entityID",
    "entityIso",
    "dataNote",
    "dataSource",
    "seriesDescription"
]

df_itu = df_itu.drop(cols_to_remove_itu)

print("CEPAL columnas:", df_cepal.columns)
print("ITU columnas:", df_itu.columns)

CEPAL columnas: ['País__ESTANDAR', 'Años__ESTANDAR', 'value']
ITU columnas: ['entityName', 'dataValue', 'dataYear']


## Paso 2: Establecimiento de un Esquema Canónico

Para garantizar la interoperabilidad técnica, se establece un **esquema de datos canónico** mediante el renombrado de columnas y la normalización de tipos. Al realizar un casting explícito a `Float64`, aseguramos la precisión matemática necesaria para los cálculos de distancias en los algoritmos de minería de datos, evitando inconsistencias entre fuentes heterogéneas.

In [143]:
df_cepal = df_cepal.rename({
    "País__ESTANDAR": "country",
    "Años__ESTANDAR": "year",
    "value": "percentage"
}).with_columns(
    pl.col("percentage").cast(pl.Float64)
).select(["country", "year", "percentage"])

df_itu = df_itu.rename({
    "entityName": "country",
    "dataYear": "year",
    "dataValue": "percentage"
}).with_columns(
    pl.col("percentage").cast(pl.Float64)
).select(["country", "year", "percentage"])

print("CEPAL:", df_cepal.head(2))
print("ITU:", df_itu.head(2))

CEPAL: shape: (2, 3)
┌───────────┬──────┬────────────┐
│ country   ┆ year ┆ percentage │
│ ---       ┆ ---  ┆ ---        │
│ str       ┆ i64  ┆ f64        │
╞═══════════╪══════╪════════════╡
│ Argentina ┆ 2016 ┆ 71.0       │
│ Argentina ┆ 2017 ┆ 74.0       │
└───────────┴──────┴────────────┘
ITU: shape: (2, 3)
┌─────────┬──────┬────────────┐
│ country ┆ year ┆ percentage │
│ ---     ┆ ---  ┆ ---        │
│ str     ┆ i64  ┆ f64        │
╞═════════╪══════╪════════════╡
│ Aruba   ┆ 2006 ┆ 36.7       │
│ Aruba   ┆ 2010 ┆ 63.0       │
└─────────┴──────┴────────────┘


## Paso 3: Resolución de Entidades 

Se realiza un proceso de **resolución de entidades** para corregir las inconsistencias léxicas entre los datasets, específicamente nombres de países en español vs. inglés. Dado que las variaciones en la escritura son una de las principales fuentes de pérdida de información en la integración de datos, normalizamos estas etiquetas para asegurar que el motor de unión reconozca correctamente a cada país como una entidad única y coherente.

In [144]:
cepal_countries = df_cepal["country"].unique().sort()
itu_countries = df_itu["country"].unique().sort()

missing_in_itu = cepal_countries.filter(~cepal_countries.is_in(itu_countries.implode()))

print("Países de CEPAL no encontrados en ITU con el mismo nombre:")
print(missing_in_itu)


MISMATCHS = {
    "Peru": "Perú",
    "Mexico": "México",
    "Panama": "Panamá",
    "Brazil" : "Brasil",
    "Bolivia (Plurinational State of)": "Bolivia (Estado Plurinacional de)",
}


df_itu = df_itu.with_columns(
    pl.col("country").replace(MISMATCHS)
)

cepal_countries = df_cepal["country"].unique()
itu_countries = df_itu["country"].unique()
matches = cepal_countries.filter(cepal_countries.is_in(itu_countries.implode()))

print(f"Países coincidentes tras normalización: {len(matches)}")

Países de CEPAL no encontrados en ITU con el mismo nombre:
shape: (5,)
Series: 'country' [str]
[
	"Bolivia (Estado Plurinacional …
	"Brasil"
	"México"
	"Panamá"
	"Perú"
]
Países coincidentes tras normalización: 14


## Paso 4: Estrategia de Fusión de Datos

Se ejecuta una unión estratégica priorizando la información de la CEPAL como nuestra **fuente de verdad primaria**. El dataset de la ITU se emplea estrictamente como fuente complementaria para cubrir vacíos temporales. Esta jerarquía es fundamental para mantener la continuidad de las series temporales y garantizar la coherencia regional, evitando sesgos que podrían surgir al mezclar metodologías de recolección distintas.

In [145]:
countries_in_cepal = df_cepal["country"].unique()
df_itu_filtered = df_itu.filter(pl.col("country").is_in(countries_in_cepal.implode()))

df_itu_complementary = df_itu_filtered.join(
    df_cepal, on=["country", "year"], how="anti"
)

df_merged = pl.concat([df_cepal, df_itu_complementary])

df_merged = df_merged.sort(["country", "year"])

print("Registros totales:", df_merged.shape[0])

Registros totales: 331


## Paso 5: Estructuración de la Matriz de Características (Pivotado)

Se transforma la estructura del dataset a un **formato de matriz de características** indispensable para el aprendizaje no supervisado. Al pivotar los años de filas a columnas, convertimos la serie temporal de cada país en un vector multidimensional. Esta configuración permite que los algoritmos de clustering calculen distancias geométricas en un espacio de "n" dimensiones (una por año) y agrupen a las naciones según sus trayectorias digitales.

In [146]:
df_final = df_merged.pivot(
    index="country",
    on="year",
    values="percentage"
)

# Asegurar que las columnas de años estén en orden ascendente (de izquierda a derecha)
# La primera columna es 'country', el resto son años
year_cols = sorted([c for c in df_final.columns if c != "country"])
df_final = df_final.select(["country"] + year_cols)

# Ordenar países alfabéticamente
df_final = df_final.sort("country")

print("Dimensiones tras el pivotado:", df_final.shape)

df_final.write_csv("pivote.csv")

Dimensiones tras el pivotado: (14, 26)


## Paso 6: Tratamiento Avanzado de Datos Faltantes e Imputación

Se finaliza el preprocesamiento con una lógica de tratamiento de nulos basada en umbrales de calidad. Si la ausencia de datos en el periodo inicial supera el 40%, se opta por una **poda estratégica** (trimming) para evitar introducir sesgos por falta de representatividad. En casos de menor carencia, se aplica un `IterativeImputer` (MICE), el cual estima los valores faltantes analizando las tendencias temporales y las relaciones entre países. Este enfoque preserva la coherencia histórica de la serie, asegurando que los resultados del clustering reflejen la evolución real del acceso tecnológico.

In [147]:
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

year_cols = [c for c in df_final.columns if c != "country"]
first_5_years = year_cols[:5]

total_cells = df_final.height * len(first_5_years)
missing_cells = df_final.select(first_5_years).null_count().sum_horizontal().item()
missing_pct = missing_cells / total_cells

print(f"Porcentaje de datos faltantes en los primeros 5 años: {missing_pct:.2%}")

if missing_pct > 0.40:
    print("Umbral del 40% superado. Realizando poda de los primeros 5 años...")
    df_final = df_final.drop(first_5_years)
else:
    print("Rellenando valores faltantes con IterativeImputer...")
    
    df_years = df_final.select(year_cols)
    
    imputer = IterativeImputer(random_state=42, min_value=1.0, max_value=100.0)
    imputed_array = imputer.fit_transform(df_years)
    
    df_imputed = pl.DataFrame(imputed_array, schema=year_cols)
    
    for col in year_cols:
        df_final = df_final.with_columns(
            pl.col(col).fill_null(df_imputed[col]).alias(col)
        )

df_final.write_csv("data_final_internet.csv")

print("Transformación completada.")
print(df_final.head())

Porcentaje de datos faltantes en los primeros 5 años: 25.71%
Rellenando valores faltantes con IterativeImputer...
Transformación completada.
shape: (5, 26)
┌───────────┬───────────┬──────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ country   ┆ 2000      ┆ 2001     ┆ 2002      ┆ … ┆ 2021      ┆ 2022      ┆ 2023      ┆ 2024      │
│ ---       ┆ ---       ┆ ---      ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ str       ┆ f64       ┆ f64      ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞═══════════╪═══════════╪══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Argentina ┆ 9.147071  ┆ 9.1      ┆ 10.0      ┆ … ┆ 87.0      ┆ 89.0      ┆ 93.380004 ┆ 93.692484 │
│ Bolivia   ┆ 1.0       ┆ 5.736992 ┆ 1.785     ┆ … ┆ 66.0      ┆ 62.625139 ┆ 66.65889  ┆ 68.73218  │
│ (Estado   ┆           ┆          ┆           ┆   ┆           ┆           ┆           ┆           │
│ Plurinaci ┆           ┆          ┆